In [ ]:
# Top 3 de franquicias por minutos HDM: viernes anterior hasta ayer.
import google.auth
import pandas as pd
import plotly.express as px
from google.cloud import bigquery
from IPython.display import display

TOP_FRANCHISES_SQL = r"""
WITH bounds AS (
  SELECT
    DATE_SUB(
      CURRENT_DATE('America/Santiago'),
      INTERVAL (MOD(EXTRACT(DAYOFWEEK FROM CURRENT_DATE('America/Santiago')), 7) + 1) DAY
    ) AS start_date,
    DATE_SUB(CURRENT_DATE('America/Santiago'), INTERVAL 1 DAY) AS end_date
), logistic_latest AS (
  SELECT
    l.peya_order_id AS order_id,
    l.vendor.vendor_code AS vendor_code,
    SAFE_DIVIDE(l.timings.avoidable_wait_time, 60.0) AS awt_min
  FROM `peya-bi-tools-pro.il_logistics.fact_logistic_orders` AS l
  CROSS JOIN bounds AS b
  WHERE l.country_code = 'cl'
    AND l.created_date_local BETWEEN b.start_date AND b.end_date
    AND l.peya_order_id IS NOT NULL
    AND l.is_preorder IS FALSE
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY l.peya_order_id ORDER BY l.audi_load_date DESC
  ) = 1
), partner AS (
  SELECT
    CAST(partner_id AS STRING) AS vendor_code,
    franchise.franchise_id AS franchise_id,
    franchise.franchise_name AS franchise_name
  FROM `peya-bi-tools-pro.il_core.dim_partner`
  WHERE country_id = 2
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY partner_id ORDER BY audi_load_date DESC, last_updated DESC
  ) = 1
), hdm_latest AS (
  SELECT
    l.order_id,
    l.vendor_code,
    o.high_demand_mode.is_hd_order AS is_hd_order,
    o.high_demand_mode.minutes_added AS minutes_added
  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_orders` AS o
  CROSS JOIN bounds AS b
  JOIN logistic_latest AS l
    ON SAFE_CAST(o.order_code AS INT64) = l.order_id
   AND o.vendor.code = l.vendor_code
  WHERE o.country_code = 'cl'
    AND o.created_date BETWEEN DATE_SUB(b.start_date, INTERVAL 1 DAY)
                           AND DATE_ADD(b.end_date, INTERVAL 1 DAY)
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY l.order_id, l.vendor_code ORDER BY o.created_at DESC
  ) = 1
), order_level AS (
  SELECT
    p.franchise_id,
    p.franchise_name,
    l.awt_min,
    CASE
      WHEN h.is_hd_order IS FALSE THEN 0.0
      WHEN h.is_hd_order IS TRUE AND h.minutes_added >= 0
      THEN CAST(h.minutes_added AS FLOAT64)
    END AS hdm_min
  FROM logistic_latest AS l
  JOIN partner AS p ON p.vendor_code = l.vendor_code
  LEFT JOIN hdm_latest AS h USING (order_id, vendor_code)
  WHERE p.franchise_id IS NOT NULL
), franchise_totals AS (
  SELECT
    franchise_id,
    ANY_VALUE(franchise_name) AS franchise_name,
    SUM(IF(hdm_min >= 0, hdm_min, NULL)) AS hdm_minutes_total,
    SUM(IF(awt_min >= 0, awt_min, NULL)) AS awt_minutes_total
  FROM order_level
  GROUP BY franchise_id
)
SELECT
  franchise_id,
  franchise_name,
  ROUND(hdm_minutes_total, 1) AS hdm_minutes_total,
  ROUND(awt_minutes_total, 1) AS awt_minutes_total,
  b.start_date,
  b.end_date
FROM franchise_totals
CROSS JOIN bounds AS b
ORDER BY hdm_minutes_total DESC
LIMIT 3
"""

_top_credentials, _ = google.auth.default(
    scopes=['https://www.googleapis.com/auth/cloud-platform'],
    quota_project_id='dhub-data-commune',
)
_top_bq = bigquery.Client(project='peya-chile', credentials=_top_credentials)
df_top_franchises = _top_bq.query(TOP_FRANCHISES_SQL).to_dataframe(
    create_bqstorage_client=False
)

if df_top_franchises.empty:
    print('No se encontraron franquicias para el período.')
else:
    start_date = pd.Timestamp(df_top_franchises.start_date.iloc[0]).date()
    end_date = pd.Timestamp(df_top_franchises.end_date.iloc[0]).date()
    print(f'Top 3 por minutos HDM · {start_date} a {end_date}')
    display(df_top_franchises)

    plot_data = df_top_franchises.melt(
        id_vars=['franchise_name'],
        value_vars=['hdm_minutes_total', 'awt_minutes_total'],
        var_name='métrica',
        value_name='minutos',
    )
    plot_data['métrica'] = plot_data['métrica'].map({
        'hdm_minutes_total': 'HDM',
        'awt_minutes_total': 'AWT',
    })
    franchise_order = df_top_franchises.franchise_name.tolist()
    fig = px.bar(
        plot_data,
        x='franchise_name',
        y='minutos',
        color='métrica',
        barmode='group',
        text_auto='.1f',
        category_orders={'franchise_name': franchise_order, 'métrica': ['HDM', 'AWT']},
        labels={'franchise_name': 'Franquicia', 'minutos': 'Minutos totales'},
        title=f'Top 3 franquicias por minutos HDM · {start_date} a {end_date}',
    )
    fig.update_layout(xaxis_title=None, legend_title_text=None)
    fig.show()


# Trigger comparison — cuándo se activan y qué efecto tienen

Este notebook reconstruye el análisis desde cero. Cada fila válida del Sheet define una comparación independiente y una **cohorte fija de vendors** para PRE y POST.

El flujo responde cuatro preguntas:

1. ¿Qué cambió en los KPIs antes y después del ajuste?
2. ¿Cómo cambió la distribución `CEIL` de AWT y de minutos HDM?
3. ¿En qué weekday, hora y bloque se concentra el HDM/AWT?
4. ¿Cuántos episodios HDM empiezan, con qué dosis y cuánto duran?

Reglas importantes:

- POST comienza en `modification_date`, incluye el día completo y termina como máximo después de 14 días completos.
- PRE tiene el mismo número de días y los mismos weekdays que POST. Se desplaza 7 días; si eso solapa POST, se desplaza 14.
- Las filas del Sheet pueden solaparse. Cada comparación es independiente y **no deben sumarse entre sí**.
- `trigger_ids_involved` etiqueta el cambio. El schema disponible no contiene trigger ID por episodio, por lo que el resultado es descriptivo, no una atribución causal a un ID.
- Los episodios se reconstruyen desde órdenes de `growth_vendor_orders`; una activación sin órdenes no es observable.


In [1]:
# Instalar únicamente dependencias ausentes (útil en Colab).
import hashlib
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    "gspread": "gspread",
    "gspread_dataframe": "gspread-dataframe",
    "google.cloud.bigquery": "google-cloud-bigquery",
    "db_dtypes": "db-dtypes",
    "plotly": "plotly",
    "pyarrow": "pyarrow",
    "nbformat": "nbformat>=4.2.0",
    "jinja2": "jinja2",
}
for module, package in REQUIRED_PACKAGES.items():
    try:
        installed = importlib.util.find_spec(module) is not None
    except ModuleNotFoundError:
        installed = False
    if not installed:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import re
from itertools import combinations

import google.auth
import gspread
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import nbformat
import plotly.io._renderers as _plotly_renderers
from google.cloud import bigquery
from IPython.display import display
from plotly.subplots import make_subplots

# Plotly puede haberse importado antes de instalar nbformat en un kernel activo.
# Reinyectarlo evita reiniciar el kernel y perder las consultas ya cargadas.
_plotly_renderers.nbformat = nbformat

PROJECT_ID = "peya-chile"
QUOTA_PROJECT_ID = "dhub-data-commune"
GOOGLE_SCOPES = [
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

# Se conservan los logins del notebook anterior: ADC funciona localmente y en Colab autenticado.
credentials, _ = google.auth.default(
    scopes=GOOGLE_SCOPES,
    quota_project_id=QUOTA_PROJECT_ID,
)
bq = bigquery.Client(project=PROJECT_ID, credentials=credentials)
gc = gspread.authorize(credentials)


In [2]:
# ----------------------------- CONFIGURACIÓN -----------------------------
INPUT_SHEET_ID = "1Df2wUV_ksnvmXpPK5uyUmBDVfrK89w5pMxN3ngMhMGI"
INPUT_WORKSHEET = "Sheet1"
OUTPUT_SHEET_ID = "1qMGldxBMdxvnc4b7e3zPiZckzZNnJWljr-2NxUBBoIs"

DAYS_TO_COMPARE = 14       # x días; debe estar entre 1 y 14.
ANALYSIS_TODAY = None      # None = fecha actual de BigQuery en America/Santiago.
SELECT_SHEET_ROWS = None   # Ejemplo: [2, 5]. None = todas las filas válidas.

EXCLUDE_PREORDERS = True
EPT_INCLUDES_HDM = True
EPISODE_SCAN_MARGIN_DAYS = 1
MAXIMUM_BYTES_BILLED = None

# Checkpoint local: evita volver a consultar al reabrir el notebook con la misma configuraciÃ³n.
USE_LOCAL_CACHE = True
FORCE_REFRESH_BIGQUERY = False
CACHE_DIRECTORY = Path(".cache") / "trigger_comparison"

# Sólo se necesita completar si el Sheet usa vendor_type pelican/no pelican.
PELICAN_CLIENT_NAMES = None  # Ejemplo: ["NOMBRE_EXACTO"]

# trigger_tag/flag seleccionan los tags que se muestran en las vistas por flag.
# No filtran la cohorte principal salvo que esta opción se cambie explícitamente.
FILTER_UNIVERSE_TO_TAG_MATCH = False

MEALPARTS = {
    "breakfast": range(8, 11),  # 08:00–10:59
    "lunch": range(12, 15),     # 12:00–14:59
    "dinner": range(19, 22),    # 19:00–21:59
}
WEEKDAY_ORDER = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
MEALPART_ORDER = ["breakfast", "lunch", "dinner"]
MIN_ORDERS_HEATMAP = 30
PLOT_BUCKET_CAP = 20       # En gráficos, CEIL >= 20 se agrupa como 20+; tablas conservan CEIL exacto.
HOURLY_MIX_WEEKDAYS = ["martes"]  # Usar WEEKDAY_ORDER para dibujar todos.
TIMELINE_MAX_VENDORS = 25
EXPORT_EPISODE_DETAIL = False  # Puede ser muy grande para el límite total de celdas de Sheets.

# Overrides no destructivos: no escriben de vuelta al Sheet de configuración.
ROW_OVERRIDES = {
    # 5: {"modification_date": "YYYY-MM-DD"},
}


## 1. Leer y validar el Sheet de cambios

Dentro de una columna, valores separados por coma son OR; entre columnas se aplica AND. `targeted_chains` incluye y `excluded_chains` excluye por `franchise_id`; la exclusión siempre prevalece. `trigger_tag` y `flag` se usan para construir cortes por tags de Vendor Monitor, no para atribuir episodios a un trigger ID.


In [3]:
def run_query(sql, params=None, *, dry_run=False, label="consulta"):
    config = bigquery.QueryJobConfig(
        query_parameters=params or [],
        dry_run=dry_run,
        use_query_cache=not dry_run,
    )
    if MAXIMUM_BYTES_BILLED is not None and not dry_run:
        config.maximum_bytes_billed = int(MAXIMUM_BYTES_BILLED)
    job = bq.query(sql, job_config=config)
    if dry_run:
        gb = (job.total_bytes_processed or 0) / 1e9
        print(f"{label} · dry run: {gb:,.3f} GB estimados")
        return job
    result = job.to_dataframe(create_bqstorage_client=False)
    gb = (job.total_bytes_processed or 0) / 1e9
    print(f"{label}: {len(result):,} filas · {gb:,.3f} GB procesados")
    return result


def clean(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def tokens(value):
    value = clean(value)
    if value.lower() in ("", "all", "todos", "todas"):
        return []
    return [item.strip() for item in value.split(",") if item.strip()]


if not 1 <= int(DAYS_TO_COMPARE) <= 14:
    raise ValueError("DAYS_TO_COMPARE debe estar entre 1 y 14.")

worksheet = gc.open_by_key(INPUT_SHEET_ID).worksheet(INPUT_WORKSHEET)
sheet_values = worksheet.get_all_values()
if not sheet_values:
    raise ValueError("El Sheet de cambios está vacío.")
headers = [value.strip() for value in sheet_values[0]]
if any(not value for value in headers) or len(headers) != len(set(headers)):
    raise ValueError("Los encabezados del Sheet deben ser únicos y no vacíos.")

df_sheet_raw = pd.DataFrame(sheet_values[1:], columns=headers)
df_sheet_raw.insert(0, "sheet_row", range(2, len(df_sheet_raw) + 2))
for row_number, changes in ROW_OVERRIDES.items():
    if row_number not in df_sheet_raw.sheet_row.values:
        raise ValueError(f"ROW_OVERRIDES referencia una fila inexistente: {row_number}")
    for column, value in changes.items():
        if column not in df_sheet_raw.columns:
            raise KeyError(f"ROW_OVERRIDES usa una columna inexistente: {column}")
        df_sheet_raw.loc[df_sheet_raw.sheet_row.eq(row_number), column] = value
if SELECT_SHEET_ROWS is not None:
    df_sheet_raw = df_sheet_raw[df_sheet_raw.sheet_row.isin(SELECT_SHEET_ROWS)].copy()

if ANALYSIS_TODAY is None:
    TODAY = pd.Timestamp(
        run_query(
            'SELECT CURRENT_DATE("America/Santiago") AS today',
            label="fecha de análisis",
        ).iloc[0].today
    ).normalize()
else:
    TODAY = pd.Timestamp(ANALYSIS_TODAY).normalize()
YESTERDAY = TODAY - pd.Timedelta(days=1)

print(f"Hoy analítico: {TODAY.date()} · último día completo: {YESTERDAY.date()}")
display(df_sheet_raw)


fecha de análisis: 1 filas · 0.000 GB procesados
Hoy analítico: 2026-09-10 · último día completo: 2026-09-09


,sheet_row,modification_date,grade,vertical,vendor_type,trigger_tag,targeted_chains,excluded_chains,flag,modification_notes,trigger_ids_involved
0,2,2026-09-04,"AAA, AA, A",restaurants,all,exclusions,,"0011r00002VoI4m,0011r00002VoHw6,0011r00002VoHw...",,se agrega una capa extra,d3fda3
1,3,2026-09-04,"B, C",restaurants,all,exclusions,,"0011r00002VoI4m,0011r00002VoHw6,0011r00002VoHw...",,se agrega una capa extra,e5b4cb
2,4,2026-09-04,"D, NA",restaurants,all,exclusions,,"0011r00002VoI4m,0011r00002VoHw6,0011r00002VoHw...",,se agrega una capa extra,04178c
3,5,2026-08-24,all,restaurants,all,exclusions,0011r00002VoI4m,,,NIU sushi: sus triggers bajan la exigencia,"26ec85, bc98ea, 56386a"


In [4]:
VENDOR_SQL = r"""
WITH attributes AS (
  SELECT
    vendor_code,
    vendor_name,
    COALESCE(NULLIF(TRIM(vendor_grade), ''), 'NA') AS vendor_grade,
    city_name,
    vertical_type,
    IFNULL(attributes.vendor_monitor_tags, ARRAY<STRING>[]) AS vendor_monitor_tags
  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_attributes_latest`
  WHERE entity_id = 'PY_CL'
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY vendor_code
    ORDER BY is_latest_record DESC, updated_date DESC, created_date DESC
  ) = 1
), partner AS (
  SELECT
    CAST(partner_id AS STRING) AS vendor_code,
    partner_name,
    franchise.franchise_id AS franchise_id,
    franchise.franchise_name AS franchise_name,
    is_online,
    is_active
  FROM `peya-bi-tools-pro.il_core.dim_partner`
  WHERE country_id = 2
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY partner_id
    ORDER BY audi_load_date DESC, last_updated DESC
  ) = 1
), vendor_client AS (
  SELECT vendor_code, latest.client.name AS client_name
  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendors`,
  UNNEST(vendor) AS latest
  WHERE entity_id = 'PY_CL' AND latest.is_latest IS TRUE
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY vendor_code ORDER BY latest.updated_at DESC
  ) = 1
)
SELECT
  a.vendor_code,
  COALESCE(p.partner_name, a.vendor_name) AS vendor_name,
  a.vendor_grade,
  a.city_name,
  a.vertical_type,
  p.franchise_id,
  p.franchise_name,
  p.vendor_code IS NOT NULL AS partner_found,
  p.is_online,
  p.is_active,
  v.client_name,
  a.vendor_monitor_tags
FROM attributes AS a
LEFT JOIN partner AS p USING (vendor_code)
LEFT JOIN vendor_client AS v USING (vendor_code)
"""

df_vendors = run_query(VENDOR_SQL, label="dimensión de vendors")
if df_vendors.vendor_code.isna().any() or df_vendors.vendor_code.duplicated().any():
    display(df_vendors[df_vendors.vendor_code.duplicated(False)])
    raise ValueError("La dimensión canónica debe tener exactamente una fila por vendor_code.")
df_vendors["vendor_code"] = df_vendors.vendor_code.astype(str)
df_vendors["franchise_id"] = df_vendors.franchise_id.astype("string")
df_vendors["vendor_monitor_tags"] = df_vendors.vendor_monitor_tags.map(
    lambda value: list(value) if isinstance(value, (list, tuple, np.ndarray)) else []
)
print(f"Dimensión canónica: {len(df_vendors):,} vendors")


dimensión de vendors: 91,286 filas · 0.036 GB procesados
Dimensión canónica: 91,286 vendors


In [5]:
REQUIRED_SHEET_COLUMNS = {
    "modification_date",
    "grade",
    "vertical",
    "vendor_type",
    "trigger_tag",
    "targeted_chains",
    "excluded_chains",
    "flag",
    "modification_notes",
    "trigger_ids_involved",
}
missing_columns = REQUIRED_SHEET_COLUMNS - set(df_sheet_raw.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas en el Sheet: {sorted(missing_columns)}")


def comparison_calendar(start_date, today, requested_days):
    start = pd.Timestamp(start_date).normalize()
    yesterday = pd.Timestamp(today).normalize() - pd.Timedelta(days=1)
    available = (yesterday - start).days + 1
    if available <= 0:
        return pd.DataFrame()
    n_days = min(int(requested_days), int(available), 14)
    post_dates = pd.date_range(start, periods=n_days, freq="D")
    shift_days = 7
    pre_start = start - pd.Timedelta(days=shift_days)
    while pre_start + pd.Timedelta(days=n_days - 1) >= start:
        shift_days += 7
        pre_start = start - pd.Timedelta(days=shift_days)
    pre_dates = pd.date_range(pre_start, periods=n_days, freq="D")
    if not np.array_equal(pre_dates.dayofweek, post_dates.dayofweek):
        raise AssertionError("PRE y POST no conservaron la misma secuencia de weekdays.")
    return pd.DataFrame({
        "pair_id": range(1, n_days + 1),
        "weekday_num": post_dates.dayofweek,
        "weekday": [WEEKDAY_ORDER[value] for value in post_dates.dayofweek],
        "pre_date": pre_dates,
        "post_date": post_dates,
        "shift_days": shift_days,
    })


def matching_tags(all_tags, selectors):
    selectors = [value.casefold() for value in selectors if clean(value)]
    if not selectors:
        return ["SIN_SELECTOR"]
    matches = sorted({
        tag for tag in all_tags
        if any(selector in str(tag).casefold() for selector in selectors)
    })
    return matches or ["SIN_TAG_MATCH"]


def salesforce_key(value):
    """Hace comparables IDs Salesforce de 15 y 18 caracteres."""
    value = clean(value)
    return value[:15] if value else ""


def build_configuration(settings, vendors):
    configs, pairs, members, errors, rules = [], [], [], [], []
    vendor_franchise_keys = vendors.franchise_id.map(salesforce_key)
    known_franchises = set(vendor_franchise_keys) - {""}
    for row in settings.to_dict("records"):
        sheet_row = int(row["sheet_row"])
        issues = []
        try:
            implementation_date = pd.to_datetime(
                clean(row["modification_date"]), format="%Y-%m-%d", errors="raise"
            ).normalize()
        except (TypeError, ValueError):
            implementation_date = pd.NaT
            issues.append("modification_date debe usar YYYY-MM-DD")

        date_pairs = (
            comparison_calendar(implementation_date, TODAY, DAYS_TO_COMPARE)
            if pd.notna(implementation_date) else pd.DataFrame()
        )
        if pd.notna(implementation_date) and date_pairs.empty:
            issues.append("no hay días POST completos hasta ayer")

        grades = tokens(row["grade"])
        verticals = tokens(row["vertical"])
        target = tokens(row["targeted_chains"])
        exclude = tokens(row["excluded_chains"])
        target_keys = {salesforce_key(value) for value in target}
        exclude_keys = {salesforce_key(value) for value in exclude}
        unknown = sorted(
            value for value in set(target) | set(exclude)
            if salesforce_key(value) not in known_franchises
        )
        if unknown:
            issues.append(f"franchise_id no encontrados: {unknown}")

        vendor_type = clean(row["vendor_type"]).casefold()
        if vendor_type not in ("", "all", "todos", "pelican", "no pelican"):
            issues.append(f"vendor_type sin mapeo: {vendor_type}")
        if vendor_type in ("pelican", "no pelican") and not PELICAN_CLIENT_NAMES:
            issues.append("completar PELICAN_CLIENT_NAMES")

        selectors = tokens(row["trigger_tag"]) + tokens(row["flag"])
        if issues:
            errors.append({"sheet_row": sheet_row, "issues": "; ".join(issues)})
            continue

        mask = pd.Series(True, index=vendors.index)
        if grades:
            mask &= vendors.vendor_grade.astype(str).isin(grades)
        if verticals:
            mask &= vendors.vertical_type.astype(str).isin(verticals)
        if target:
            mask &= vendor_franchise_keys.isin(target_keys)
        if exclude:
            mask &= ~vendor_franchise_keys.isin(exclude_keys)
        if vendor_type in ("pelican", "no pelican"):
            pelican = vendors.client_name.isin(PELICAN_CLIENT_NAMES)
            mask &= pelican if vendor_type == "pelican" else ~pelican

        selected = vendors.loc[mask].copy()
        selected["analysis_flags"] = selected.vendor_monitor_tags.map(
            lambda all_tags: matching_tags(all_tags, selectors)
        )
        if FILTER_UNIVERSE_TO_TAG_MATCH and selectors:
            selected = selected[
                ~selected.analysis_flags.map(lambda values: values == ["SIN_TAG_MATCH"])
            ].copy()
        if selected.empty:
            errors.append({"sheet_row": sheet_row, "issues": "universo vacío tras aplicar filtros"})
            continue

        comparison_id = f"change_row_{sheet_row}"
        notes = clean(row["modification_notes"]) or "cambio de triggers"
        label = f"{implementation_date:%Y-%m-%d} · {notes} · fila {sheet_row}"
        config = {
            **row,
            "comparison_id": comparison_id,
            "comparison_label": label,
            "implementation_date": implementation_date,
            "days_per_period": len(date_pairs),
            "pre_start": date_pairs.pre_date.min(),
            "pre_end": date_pairs.pre_date.max(),
            "post_start": date_pairs.post_date.min(),
            "post_end": date_pairs.post_date.max(),
            "shift_days": int(date_pairs.shift_days.iloc[0]),
            "eligible_vendors": selected.vendor_code.nunique(),
            "eligible_franchises": selected.franchise_id.nunique(dropna=True),
        }
        configs.append(config)
        pairs.append(date_pairs.assign(comparison_id=comparison_id))
        members.append(selected.assign(comparison_id=comparison_id))

        for action, ids in (("include", target), ("exclude", exclude)):
            for franchise_id in ids:
                match = vendors[vendor_franchise_keys.eq(salesforce_key(franchise_id))]
                rules.append({
                    "comparison_id": comparison_id,
                    "sheet_row": sheet_row,
                    "action": action,
                    "sheet_franchise_id": franchise_id,
                    "franchise_id": (
                        match.franchise_id.dropna().iloc[0]
                        if not match.franchise_id.dropna().empty else None
                    ),
                    "franchise_name": (
                        match.franchise_name.dropna().iloc[0]
                        if not match.franchise_name.dropna().empty else None
                    ),
                    "vendors_in_dimension": match.vendor_code.nunique(),
                })

    return (
        pd.DataFrame(configs),
        pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame(),
        pd.concat(members, ignore_index=True) if members else pd.DataFrame(),
        pd.DataFrame(errors, columns=["sheet_row", "issues"]),
        pd.DataFrame(rules, columns=[
            "comparison_id", "sheet_row", "action", "sheet_franchise_id",
            "franchise_id", "franchise_name", "vendors_in_dimension",
        ]),
    )


df_config, df_date_pairs, df_universe, df_config_errors, df_franchise_rules = build_configuration(
    df_sheet_raw, df_vendors
)
if df_config.empty:
    display(df_config_errors)
    raise ValueError("No quedaron filas válidas del Sheet.")

df_calendar = pd.concat([
    df_date_pairs.rename(columns={"pre_date": "order_date"}).assign(period="PRE"),
    df_date_pairs.rename(columns={"post_date": "order_date"}).assign(period="POST"),
], ignore_index=True)[
    ["comparison_id", "period", "pair_id", "weekday_num", "weekday", "order_date"]
]
df_calendar["order_date"] = pd.to_datetime(df_calendar.order_date)

selected_franchise_columns = [
    "comparison_id", "franchise_id", "franchise_name"
]
df_selected_franchises = (
    df_universe[selected_franchise_columns]
    .drop_duplicates()
    .sort_values(["comparison_id", "franchise_name"], na_position="last")
)
df_universe_audit = df_config[[
    "comparison_id", "sheet_row", "comparison_label", "implementation_date",
    "pre_start", "pre_end", "post_start", "post_end", "days_per_period",
    "shift_days", "eligible_vendors", "eligible_franchises", "targeted_chains",
    "excluded_chains", "grade", "vertical", "vendor_type", "trigger_tag",
    "flag", "trigger_ids_involved",
]].copy()

overlap_rows = []
member_sets = {
    key: set(group.vendor_code)
    for key, group in df_universe.groupby("comparison_id")
}
for left, right in combinations(member_sets, 2):
    overlap_rows.append({
        "comparison_left": left,
        "comparison_right": right,
        "overlapping_vendors": len(member_sets[left] & member_sets[right]),
    })
df_overlap_audit = pd.DataFrame(overlap_rows)

display(df_config_errors)
display(df_universe_audit)
display(df_franchise_rules)
display(df_date_pairs)
display(df_selected_franchises)
display(df_overlap_audit)


,sheet_row,issues


,comparison_id,sheet_row,comparison_label,implementation_date,pre_start,pre_end,post_start,post_end,days_per_period,shift_days,eligible_vendors,eligible_franchises,targeted_chains,excluded_chains,grade,vertical,vendor_type,trigger_tag,flag,trigger_ids_involved
0,change_row_2,2,2026-09-04 · se agrega una capa extra · fila 2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,7,538,70,,"0011r00002VoI4m,0011r00002VoHw6,0011r00002VoHw...","AAA, AA, A",restaurants,all,exclusions,,d3fda3
1,change_row_3,3,2026-09-04 · se agrega una capa extra · fila 3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,7,4184,222,,"0011r00002VoI4m,0011r00002VoHw6,0011r00002VoHw...","B, C",restaurants,all,exclusions,,e5b4cb
2,change_row_4,4,2026-09-04 · se agrega una capa extra · fila 4,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,7,65601,540,,"0011r00002VoI4m,0011r00002VoHw6,0011r00002VoHw...","D, NA",restaurants,all,exclusions,,04178c
3,change_row_5,5,2026-08-24 · NIU sushi: sus triggers bajan la ...,2026-08-24,2026-08-10,2026-08-23,2026-08-24,2026-09-06,14,14,63,1,0011r00002VoI4m,,all,restaurants,all,exclusions,,"26ec85, bc98ea, 56386a"


,comparison_id,sheet_row,action,sheet_franchise_id,franchise_id,franchise_name,vendors_in_dimension
0,change_row_2,2,exclude,0011r00002VoI4m,0011r00002VoI4mAAF,NIU,68
1,change_row_2,2,exclude,0011r00002VoHw6,0011r00002VoHw6AAF,Papa John's,330
2,change_row_2,2,exclude,0011r00002VoHw2,0011r00002VoHw2AAF,KFC,348
3,change_row_2,2,exclude,0016900002Zmgdu,0016900002ZmgduAAB,Under Pizzas,49
4,change_row_2,2,exclude,0011r00002VoISN,0011r00002VoISNAA3,Melt Pizzas,57
5,change_row_2,2,exclude,0011r00002VoHw4,0011r00002VoHw4AAF,McDonald's,376
6,change_row_2,2,exclude,0011r00002VoHvw,0011r00002VoHvwAAF,Burger King,97
7,change_row_2,2,exclude,0011r00002VoHwC,0011r00002VoHwCAAV,Wendy's,103
8,change_row_3,3,exclude,0011r00002VoI4m,0011r00002VoI4mAAF,NIU,68
9,change_row_3,3,exclude,0011r00002VoHw6,0011r00002VoHw6AAF,Papa John's,330


,pair_id,weekday_num,weekday,pre_date,post_date,shift_days,comparison_id
0,1,4,viernes,2026-08-28,2026-09-04,7,change_row_2
1,2,5,sábado,2026-08-29,2026-09-05,7,change_row_2
2,3,6,domingo,2026-08-30,2026-09-06,7,change_row_2
3,4,0,lunes,2026-08-31,2026-09-07,7,change_row_2
4,5,1,martes,2026-09-01,2026-09-08,7,change_row_2
5,6,2,miércoles,2026-09-02,2026-09-09,7,change_row_2
6,1,4,viernes,2026-08-28,2026-09-04,7,change_row_3
7,2,5,sábado,2026-08-29,2026-09-05,7,change_row_3
8,3,6,domingo,2026-08-30,2026-09-06,7,change_row_3
9,4,0,lunes,2026-08-31,2026-09-07,7,change_row_3


,comparison_id,franchise_id,franchise_name
226,change_row_2,0011r00002VoIN5AAN,Affogato Waffles Helados Y Cafe
6,change_row_2,0011r00002VoHwkAAF,Alemán Experto
309,change_row_2,0016900002nctNWAAY,Barrio Chicken
91,change_row_2,0011r00002XHwEdAAL,Bendito Sushi
305,change_row_2,0011r00002Xc4QbAAJ,Buffet (mechado)
...,...,...,...
5502,change_row_4,0011r00002VoIGmAAN,Yogen Fruz
16064,change_row_4,0011r00002VoINAAA3,Yogurtlife
33574,change_row_4,0016900002dYyQJAA0,Yutakana
4722,change_row_4,<NA>,NaN


,comparison_left,comparison_right,overlapping_vendors
0,change_row_2,change_row_3,0
1,change_row_2,change_row_4,0
2,change_row_2,change_row_5,0
3,change_row_3,change_row_4,0
4,change_row_3,change_row_5,0
5,change_row_4,change_row_5,0


## 2. Dos fuentes canónicas: órdenes y episodios

La primera query parte de una fila canónica por orden, evita expandir `deliveries` y deduplica HDM por `order_id + vendor_code`. Conserva conflictos y cobertura en vez de convertir faltantes silenciosamente a cero.

La segunda query separa episodios: deduplica las órdenes que observaron el mismo `vendor_code + enabled_at`. `duration` se divide por 60 porque la fuente lo almacena en segundos; además se contrasta contra `disabled_at - enabled_at`.


In [6]:
ORDER_AGG_SQL = r"""
WITH logistic_latest AS (
  SELECT l.*
  FROM `peya-bi-tools-pro.il_logistics.fact_logistic_orders` AS l
  WHERE l.country_code = 'cl'
    AND l.created_date_local BETWEEN @min_date AND @max_date
    AND l.created_date_local IN UNNEST(@analysis_dates)
    AND l.vendor.vendor_code IN UNNEST(@vendor_codes)
    AND l.peya_order_id IS NOT NULL
    AND (NOT @exclude_preorders OR l.is_preorder IS FALSE)
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY l.peya_order_id ORDER BY l.audi_load_date DESC
  ) = 1
), logistic AS (
  SELECT
    l.peya_order_id AS order_id,
    l.vendor.vendor_code AS vendor_code,
    DATE(l.created_date_local) AS order_date,
    EXTRACT(HOUR FROM l.created_at_local) AS order_hour,
    SAFE_DIVIDE(l.estimated_prep_time, 60.0) AS ept_total_min,
    SAFE_DIVIDE(l.timings.avoidable_wait_time, 60.0) AS awt_min,
    TIMESTAMP_DIFF(l.food_is_ready_at, l.created_at, SECOND) / 60.0 AS fir_min,
    (SELECT COUNT(*) FROM UNNEST(l.deliveries) AS d WHERE d.is_primary) AS primary_deliveries,
    CASE
      WHEN (SELECT COUNT(*) FROM UNNEST(l.deliveries) AS d WHERE d.is_primary) = 1
      THEN (
        SELECT DATETIME_DIFF(
          d.rider_picked_up_at_local, l.sent_to_vendor_at_local, SECOND
        ) / 60.0
        FROM UNNEST(l.deliveries) AS d
        WHERE d.is_primary
        LIMIT 1
      )
    END AS pickup_min
  FROM logistic_latest AS l
), hdm_raw AS (
  SELECT
    SAFE_CAST(o.order_code AS INT64) AS order_id,
    o.vendor.code AS vendor_code,
    COUNT(*) AS source_rows,
    COUNT(DISTINCT TO_JSON_STRING(STRUCT(
      o.high_demand_mode.is_hd_order,
      o.high_demand_mode.minutes_added,
      o.high_demand_mode.author,
      o.high_demand_mode.enabled_at,
      o.high_demand_mode.disabled_at,
      o.high_demand_mode.duration
    ))) AS variants,
    ARRAY_AGG(STRUCT(
      o.high_demand_mode.is_hd_order AS is_hd_order,
      o.high_demand_mode.minutes_added AS minutes_added,
      o.high_demand_mode.author AS author
    ) ORDER BY o.created_at DESC LIMIT 1)[OFFSET(0)] AS state
  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_orders` AS o
  WHERE o.country_code = 'cl'
    AND o.created_date BETWEEN DATE_SUB(@min_date, INTERVAL 1 DAY)
                           AND DATE_ADD(@max_date, INTERVAL 1 DAY)
    AND o.vendor.code IN UNNEST(@vendor_codes)
  GROUP BY 1, 2
), reason_metrics AS (
  SELECT
    r.order_id,
    SUM(r.TOTAL_ORDERS) AS reason_base,
    SUM(
      r.TR_HIGH_PREP_TIME
      + r.TR_HIGH_PREP_TIME_AND_DISTANCE / 2
      + r.TR_HIGH_PREP_TIME_AND_DISTANCE_COMBINATION / 2
    ) AS high_prep_impact,
    SUM(IF(
      IFNULL(r.undispatch_reason, '') != 'Late Order Preparation',
      r.TR_STAFFING_AND_PARTNER_PERFO / 2
        + r.TR_STAFFING_RIDER_AND_PARTNER_PERFO / 3
        + r.TR_RIDER_AND_PARTNER_PERFO / 2
        + r.TR_PARTNER_PERFO,
      0
    )) AS awt10_impact,
    SUM(r.PARTNER_PERFORMANCE) AS partner_performance_impact
  FROM `peya-delivery-and-support.automated_tables_reports.non_seamless_reasons` AS r
  WHERE r.country_name = 'Chile'
    AND r.created_date_local BETWEEN @min_date AND @max_date
  GROUP BY 1
), seamless AS (
  SELECT
    platform_order_code AS order_id,
    MAX(is_slow_order) AS is_slow_order,
    LOGICAL_OR(non_seamless_order) AS non_seamless_order
  FROM `peya-datamarts-pro.dm_fulfillment.non_seamless_delivery_order_level`
  WHERE country_name = 'Chile'
    AND created_date_local BETWEEN @min_date AND @max_date
  GROUP BY 1
), order_level AS (
  SELECT
    l.*,
    h.source_rows,
    h.variants,
    CASE
      WHEN h.variants = 1 AND h.state.is_hd_order IS FALSE THEN 0.0
      WHEN h.variants = 1 AND h.state.is_hd_order IS TRUE
        AND h.state.minutes_added >= 0
      THEN CAST(h.state.minutes_added AS FLOAT64)
    END AS hdm_min,
    CASE WHEN h.variants = 1 THEN h.state.author END AS hdm_author,
    r.order_id IS NOT NULL AS reason_matched,
    r.reason_base,
    r.high_prep_impact,
    r.awt10_impact,
    r.partner_performance_impact,
    s.is_slow_order,
    s.non_seamless_order
  FROM logistic AS l
  LEFT JOIN hdm_raw AS h
    ON h.order_id = l.order_id AND h.vendor_code = l.vendor_code
  LEFT JOIN reason_metrics AS r ON r.order_id = l.order_id
  LEFT JOIN seamless AS s ON s.order_id = l.order_id
), enriched AS (
  SELECT
    *,
    CASE
      WHEN awt_min >= 0 THEN CAST(CEIL(awt_min) AS INT64)
    END AS awt_ceil_min,
    CASE
      WHEN hdm_min >= 0 THEN CAST(CEIL(hdm_min) AS INT64)
    END AS hdm_ceil_min,
    IF(@ept_includes_hdm, ept_total_min - hdm_min, ept_total_min) AS ept_without_hdm_min,
    ept_total_min IS NOT NULL AND hdm_min IS NOT NULL AND awt_min IS NOT NULL AS stack_known
  FROM order_level
)
SELECT
  vendor_code,
  order_date,
  order_hour,
  awt_ceil_min,
  hdm_ceil_min,
  IF(hdm_min > 0, hdm_author, 'SIN_HDM') AS hdm_author,
  COUNT(*) AS orders,
  COUNT(ept_total_min) AS ept_known_orders,
  SUM(ept_total_min) AS ept_total_minutes,
  COUNT(ept_without_hdm_min) AS ept_base_known_orders,
  SUM(ept_without_hdm_min) AS ept_base_minutes,
  COUNT(awt_min) AS awt_known_orders,
  SUM(IF(awt_min >= 0, awt_min, NULL)) AS awt_minutes,
  COUNTIF(awt_min > 0) AS awt_positive_orders,
  COUNTIF(awt_min > 10) AS awt_gt10_orders,
  COUNTIF(awt_min < 0) AS negative_awt_orders,
  COUNT(hdm_min) AS hdm_known_orders,
  COUNTIF(hdm_min > 0) AS hdm_orders,
  SUM(IF(hdm_min >= 0, hdm_min, NULL)) AS hdm_minutes,
  COUNTIF(hdm_min > 0 AND hdm_author = 'log_vendor_monitor') AS vm_hdm_orders,
  SUM(IF(hdm_min > 0 AND hdm_author = 'log_vendor_monitor', hdm_min, 0)) AS vm_hdm_minutes,
  COUNTIF(source_rows IS NULL) AS missing_hdm_source_orders,
  COUNTIF(variants > 1) AS conflicting_hdm_orders,
  COUNTIF(ept_without_hdm_min < 0) AS negative_ept_base_orders,
  COUNTIF(primary_deliveries > 1) AS multiple_primary_orders,
  COUNTIF(stack_known) AS stack_known_orders,
  SUM(IF(stack_known, ept_without_hdm_min, NULL)) AS stack_ept_base_minutes,
  SUM(IF(stack_known, hdm_min, NULL)) AS stack_hdm_minutes,
  SUM(IF(stack_known, awt_min, NULL)) AS stack_awt_minutes,
  COUNT(fir_min) AS fir_known_orders,
  SUM(fir_min) AS fir_minutes,
  COUNT(pickup_min) AS pickup_known_orders,
  SUM(pickup_min) AS pickup_minutes,
  COUNT(is_slow_order) AS slow_known_orders,
  COUNTIF(is_slow_order = 1) AS slow_orders,
  COUNT(non_seamless_order) AS seamless_known_orders,
  COUNTIF(non_seamless_order IS FALSE) AS seamless_orders,
  COUNTIF(reason_matched) AS reason_matched_orders,
  SUM(reason_base) AS reason_base,
  SUM(high_prep_impact) AS high_prep_impact,
  SUM(awt10_impact) AS awt10_impact,
  SUM(partner_performance_impact) AS partner_performance_impact
FROM enriched
GROUP BY 1, 2, 3, 4, 5, 6
"""


In [7]:
EPISODE_SQL = r"""
WITH raw AS (
  SELECT
    o.vendor.code AS vendor_code,
    o.order_code,
    o.created_at AS order_created_at,
    o.high_demand_mode.author AS author,
    o.high_demand_mode.enabled_at AS enabled_at,
    o.high_demand_mode.disabled_at AS disabled_at,
    o.high_demand_mode.minutes_added AS minutes_added,
    o.high_demand_mode.duration AS duration_seconds
  FROM `fulfillment-dwh-production.curated_data_shared_vendor.growth_vendor_orders` AS o
  WHERE o.country_code = 'cl'
    AND o.created_date BETWEEN @episode_scan_start AND @episode_scan_end
    AND o.vendor.code IN UNNEST(@vendor_codes)
    AND o.high_demand_mode.is_hd_order IS TRUE
    AND o.high_demand_mode.enabled_at IS NOT NULL
), episodes AS (
  SELECT
    vendor_code,
    enabled_at,
    COUNT(DISTINCT order_code) AS observed_orders,
    COUNT(DISTINCT minutes_added) AS dose_variants,
    COUNT(DISTINCT author) AS author_variants,
    MIN(order_created_at) AS first_observed_order_at,
    MAX(order_created_at) AS last_observed_order_at,
    MAX(disabled_at) AS disabled_at,
    MAX(duration_seconds) AS duration_seconds,
    ANY_VALUE(minutes_added) AS minutes_added,
    ANY_VALUE(author) AS author
  FROM raw
  GROUP BY 1, 2
)
SELECT
  vendor_code,
  enabled_at,
  disabled_at,
  DATETIME(enabled_at, 'America/Santiago') AS enabled_at_local,
  DATE(DATETIME(enabled_at, 'America/Santiago')) AS enabled_date_local,
  EXTRACT(HOUR FROM DATETIME(enabled_at, 'America/Santiago')) AS enabled_hour_local,
  observed_orders,
  dose_variants,
  author_variants,
  IF(dose_variants = 1, minutes_added, NULL) AS minutes_added,
  IF(author_variants = 1, author, 'MULTIPLE_AUTHORS') AS author,
  SAFE_DIVIDE(duration_seconds, 60.0) AS reported_duration_min,
  TIMESTAMP_DIFF(disabled_at, enabled_at, SECOND) / 60.0 AS timestamp_duration_min,
  disabled_at IS NULL AS right_censored,
  first_observed_order_at,
  last_observed_order_at
FROM episodes
"""


In [8]:
def query_parameters():
    vendor_codes = sorted(df_universe.vendor_code.astype(str).unique())
    analysis_dates = sorted(pd.to_datetime(df_calendar.order_date).dt.date.unique())
    min_date, max_date = min(analysis_dates), max(analysis_dates)
    order_params = [
        bigquery.ArrayQueryParameter("analysis_dates", "DATE", analysis_dates),
        bigquery.ArrayQueryParameter("vendor_codes", "STRING", vendor_codes),
        bigquery.ScalarQueryParameter("min_date", "DATE", min_date),
        bigquery.ScalarQueryParameter("max_date", "DATE", max_date),
        bigquery.ScalarQueryParameter("exclude_preorders", "BOOL", EXCLUDE_PREORDERS),
        bigquery.ScalarQueryParameter("ept_includes_hdm", "BOOL", EPT_INCLUDES_HDM),
    ]
    episode_params = [
        bigquery.ArrayQueryParameter("vendor_codes", "STRING", vendor_codes),
        bigquery.ScalarQueryParameter(
            "episode_scan_start", "DATE",
            min_date - pd.Timedelta(days=EPISODE_SCAN_MARGIN_DAYS),
        ),
        bigquery.ScalarQueryParameter(
            "episode_scan_end", "DATE",
            max_date + pd.Timedelta(days=EPISODE_SCAN_MARGIN_DAYS),
        ),
    ]
    return order_params, episode_params


def current_query_cache_key():
    payload = {
        "cache_schema_version": 1,
        "project_id": PROJECT_ID,
        "vendor_codes": sorted(df_universe.vendor_code.astype(str).unique()),
        "analysis_dates": sorted(
            pd.to_datetime(df_calendar.order_date).dt.strftime("%Y-%m-%d").unique()
        ),
        "exclude_preorders": EXCLUDE_PREORDERS,
        "ept_includes_hdm": EPT_INCLUDES_HDM,
        "episode_scan_margin_days": EPISODE_SCAN_MARGIN_DAYS,
        "order_sql": ORDER_AGG_SQL,
        "episode_sql": EPISODE_SQL,
    }
    serialized = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()[:16]


order_params, episode_params = query_parameters()
query_cache_key = current_query_cache_key()
order_cache_path = CACHE_DIRECTORY / f"orders_{query_cache_key}.parquet"
episode_cache_path = CACHE_DIRECTORY / f"episodes_{query_cache_key}.parquet"

frames_in_memory = all(
    name in globals() and isinstance(globals()[name], pd.DataFrame)
    for name in ["df_order_agg", "df_episodes_raw"]
)
memory_cache_key = globals().get("_trigger_query_cache_key")
reuse_memory = (
    not FORCE_REFRESH_BIGQUERY
    and frames_in_memory
    and memory_cache_key in (None, query_cache_key)
)
cache_source = None

if reuse_memory:
    cache_source = "memoria de este kernel"
    if memory_cache_key is None:
        print("Reutilizando la consulta ya terminada antes de habilitar el checkpoint.")
elif (
    USE_LOCAL_CACHE
    and not FORCE_REFRESH_BIGQUERY
    and order_cache_path.exists()
    and episode_cache_path.exists()
):
    try:
        df_order_agg = pd.read_parquet(order_cache_path)
        df_episodes_raw = pd.read_parquet(episode_cache_path)
        cache_source = f"checkpoint local {query_cache_key}"
    except Exception as error:
        print(f"Checkpoint ilegible ({error}); se regenerará desde BigQuery.")

if cache_source is None:
    run_query(ORDER_AGG_SQL, order_params, dry_run=True, label="órdenes agregadas")
    run_query(EPISODE_SQL, episode_params, dry_run=True, label="episodios HDM")
    df_order_agg = run_query(ORDER_AGG_SQL, order_params, label="órdenes agregadas")
    df_episodes_raw = run_query(EPISODE_SQL, episode_params, label="episodios HDM")
    cache_source = "BigQuery"

df_order_agg["vendor_code"] = df_order_agg.vendor_code.astype(str)
df_order_agg["order_date"] = pd.to_datetime(df_order_agg.order_date)
df_episodes_raw["vendor_code"] = df_episodes_raw.vendor_code.astype(str)
df_episodes_raw["enabled_date_local"] = pd.to_datetime(df_episodes_raw.enabled_date_local)
if df_order_agg.duplicated([
    "vendor_code", "order_date", "order_hour", "awt_ceil_min",
    "hdm_ceil_min", "hdm_author",
]).any():
    raise ValueError("La query de órdenes devolvió un grain duplicado.")
if df_episodes_raw.duplicated(["vendor_code", "enabled_at"]).any():
    raise ValueError("La query de episodios devolvió vendor_code + enabled_at duplicado.")

if USE_LOCAL_CACHE and cache_source != f"checkpoint local {query_cache_key}":
    CACHE_DIRECTORY.mkdir(parents=True, exist_ok=True)
    df_order_agg.to_parquet(order_cache_path, index=False)
    df_episodes_raw.to_parquet(episode_cache_path, index=False)
    print(f"Checkpoint guardado: {query_cache_key}")
_trigger_query_cache_key = query_cache_key
print(
    f"Fuente: {cache_source} · {len(df_order_agg):,} filas de órdenes · "
    f"{len(df_episodes_raw):,} episodios"
)


órdenes agregadas · dry run: 15.795 GB estimados
episodios HDM · dry run: 13.639 GB estimados
órdenes agregadas: 1,501,177 filas · 15.795 GB procesados
episodios HDM: 128,751 filas · 13.639 GB procesados


## 3. Etiquetar cada observación con su comparación

La membresía actual se congela y se aplica igual a PRE y POST. Los tags se mantienen como arrays hasta las vistas por flag, evitando multiplicar órdenes en los totales.


In [9]:
MEMBER_COLUMNS = [
    "comparison_id", "vendor_code", "vendor_name", "vendor_grade", "city_name",
    "vertical_type", "franchise_id", "franchise_name", "client_name", "analysis_flags",
]
df_order_analysis = (
    df_order_agg
    .merge(df_universe[MEMBER_COLUMNS], on="vendor_code", how="inner", validate="many_to_many")
    .merge(
        df_calendar,
        left_on=["comparison_id", "order_date"],
        right_on=["comparison_id", "order_date"],
        how="inner",
        validate="many_to_one",
    )
)

def mealpart_from_hour(hour):
    if pd.isna(hour):
        return "other"
    hour = int(hour)
    for name, hours in MEALPARTS.items():
        if hour in hours:
            return name
    return "other"

df_order_analysis["mealpart"] = df_order_analysis.order_hour.map(mealpart_from_hour)
df_order_analysis["weekday"] = pd.Categorical(
    df_order_analysis.weekday, WEEKDAY_ORDER, ordered=True
)

df_episode_analysis = (
    df_episodes_raw
    .merge(df_universe[MEMBER_COLUMNS], on="vendor_code", how="inner", validate="many_to_many")
    .merge(
        df_calendar,
        left_on=["comparison_id", "enabled_date_local"],
        right_on=["comparison_id", "order_date"],
        how="inner",
        validate="many_to_one",
    )
)
df_episode_analysis["mealpart"] = df_episode_analysis.enabled_hour_local.map(mealpart_from_hour)
df_episode_analysis["duration_min"] = df_episode_analysis.timestamp_duration_min.fillna(
    df_episode_analysis.reported_duration_min
)
df_episode_analysis["duration_difference_min"] = (
    df_episode_analysis.timestamp_duration_min - df_episode_analysis.reported_duration_min
).abs()
df_episode_analysis["is_vendor_monitor"] = df_episode_analysis.author.eq("log_vendor_monitor")
df_episode_analysis["weekday"] = pd.Categorical(
    df_episode_analysis.weekday, WEEKDAY_ORDER, ordered=True
)

print(f"Grain analítico de órdenes: {len(df_order_analysis):,} filas agregadas")
print(f"Episodios observados y elegibles: {len(df_episode_analysis):,}")


Grain analítico de órdenes: 635,418 filas agregadas
Episodios observados y elegibles: 61,827


## 4. Tablas comparativas (antes “Holy Tables”)

Los promedios y shares se recalculan como suma de numeradores / suma de denominadores. `hdm_frequency_pct`, `hdm_intensity_min` y `hdm_load_min_per_order` separan frecuencia, dosis entre afectados y carga promedio sobre el universo.


In [10]:
ADDITIVE_COLUMNS = [
    "orders", "ept_known_orders", "ept_total_minutes", "ept_base_known_orders",
    "ept_base_minutes", "awt_known_orders", "awt_minutes", "awt_positive_orders",
    "awt_gt10_orders", "negative_awt_orders", "hdm_known_orders", "hdm_orders",
    "hdm_minutes", "vm_hdm_orders", "vm_hdm_minutes", "missing_hdm_source_orders",
    "conflicting_hdm_orders", "negative_ept_base_orders", "multiple_primary_orders",
    "stack_known_orders", "stack_ept_base_minutes", "stack_hdm_minutes",
    "stack_awt_minutes", "fir_known_orders", "fir_minutes", "pickup_known_orders",
    "pickup_minutes", "slow_known_orders", "slow_orders", "seamless_known_orders",
    "seamless_orders", "reason_matched_orders", "reason_base", "high_prep_impact",
    "awt10_impact", "partner_performance_impact",
]


def safe_divide(numerator, denominator, multiplier=1.0):
    return multiplier * numerator.astype(float).div(
        denominator.astype(float).replace(0, np.nan)
    )


def aggregate_orders(data, dimensions):
    group_keys = list(dimensions) + ["period"]
    result = (
        data.groupby(group_keys, observed=True, dropna=False)
        .agg(
            **{column: (column, "sum") for column in ADDITIVE_COLUMNS},
            active_vendors=("vendor_code", "nunique"),
        )
        .reset_index()
    )
    result["ept_total_avg_min"] = safe_divide(result.ept_total_minutes, result.ept_known_orders)
    result["ept_without_hdm_avg_min"] = safe_divide(result.ept_base_minutes, result.ept_base_known_orders)
    result["awt_avg_min"] = safe_divide(result.awt_minutes, result.awt_known_orders)
    result["awt_positive_share_pct"] = safe_divide(result.awt_positive_orders, result.awt_known_orders, 100)
    result["awt_gt10_share_pct"] = safe_divide(result.awt_gt10_orders, result.awt_known_orders, 100)
    result["hdm_source_coverage_pct"] = safe_divide(result.hdm_known_orders, result.orders, 100)
    result["hdm_frequency_pct"] = safe_divide(result.hdm_orders, result.orders, 100)
    result["hdm_frequency_known_pct"] = safe_divide(result.hdm_orders, result.hdm_known_orders, 100)
    result["hdm_intensity_min"] = safe_divide(result.hdm_minutes, result.hdm_orders)
    result["hdm_load_min_per_order"] = safe_divide(result.hdm_minutes, result.orders)
    result["hdm_minutes_per_100_orders"] = safe_divide(result.hdm_minutes, result.orders, 100)
    result["vm_hdm_frequency_pct"] = safe_divide(result.vm_hdm_orders, result.orders, 100)
    result["fir_avg_min"] = safe_divide(result.fir_minutes, result.fir_known_orders)
    result["pickup_avg_min"] = safe_divide(result.pickup_minutes, result.pickup_known_orders)
    result["slow_share_pct"] = safe_divide(result.slow_orders, result.slow_known_orders, 100)
    result["seamless_share_pct"] = safe_divide(result.seamless_orders, result.seamless_known_orders, 100)
    result["high_prep_share_pct"] = safe_divide(result.high_prep_impact, result.reason_base, 100)
    result["awt10_impact_share_pct"] = safe_divide(result.awt10_impact, result.reason_base, 100)
    result["partner_performance_share_pct"] = safe_divide(
        result.partner_performance_impact, result.reason_base, 100
    )
    result["reason_coverage_pct"] = safe_divide(result.reason_matched_orders, result.orders, 100)
    result["stack_coverage_pct"] = safe_divide(result.stack_known_orders, result.orders, 100)
    result["stack_ept_base_avg_min"] = safe_divide(
        result.stack_ept_base_minutes, result.stack_known_orders
    )
    result["stack_hdm_avg_min"] = safe_divide(result.stack_hdm_minutes, result.stack_known_orders)
    result["stack_awt_avg_min"] = safe_divide(result.stack_awt_minutes, result.stack_known_orders)
    return result


def aggregate_episodes(data, dimensions):
    keys = list(dimensions) + ["period"]
    if data.empty:
        return pd.DataFrame(columns=keys)

    def summarize_group(group):
        vm = group[group.is_vendor_monitor]
        duration = pd.to_numeric(vm.duration_min, errors="coerce").dropna()
        return pd.Series({
            "hdm_episode_starts": len(group),
            "vm_episode_starts": len(vm),
            "vendors_with_episode": group.vendor_code.nunique(),
            "vm_episode_duration_avg_min": duration.mean(),
            "vm_episode_duration_p50_min": duration.quantile(0.50),
            "vm_episode_duration_p90_min": duration.quantile(0.90),
            "vm_episode_minutes_total": duration.sum(min_count=1),
            "vm_episode_dose_avg_min": pd.to_numeric(vm.minutes_added, errors="coerce").mean(),
            "right_censored_episodes": group.right_censored.fillna(False).sum(),
            "episode_dose_conflicts": group.dose_variants.gt(1).sum(),
            "episode_duration_mismatches": group.duration_difference_min.gt(0.1).sum(),
        })

    return (
        data.groupby(keys, observed=True, dropna=False)
        .apply(summarize_group, include_groups=False)
        .reset_index()
    )


SUMMARY_METRICS = [
    "orders", "active_vendors", "ept_total_avg_min", "ept_without_hdm_avg_min",
    "awt_avg_min", "awt_positive_share_pct", "awt_gt10_share_pct",
    "hdm_frequency_pct", "hdm_frequency_known_pct", "hdm_intensity_min",
    "hdm_load_min_per_order", "hdm_minutes_per_100_orders", "hdm_minutes",
    "vm_hdm_frequency_pct", "fir_avg_min", "pickup_avg_min", "slow_share_pct",
    "seamless_share_pct", "high_prep_share_pct", "awt10_impact_share_pct",
    "partner_performance_share_pct", "hdm_source_coverage_pct", "reason_coverage_pct",
    "stack_coverage_pct", "hdm_episode_starts", "vm_episode_starts",
    "vm_episode_starts_per_1000_orders", "vm_episode_duration_avg_min",
    "vm_episode_duration_p50_min", "vm_episode_duration_p90_min",
    "vm_episode_minutes_total", "vm_episode_dose_avg_min",
]


def build_summary(data, episodes, dimensions):
    result = aggregate_orders(data, dimensions)
    episode_result = aggregate_episodes(episodes, dimensions)
    if not episode_result.empty:
        result = result.merge(
            episode_result, on=list(dimensions) + ["period"], how="left", validate="one_to_one"
        )
    episode_count_columns = [
        "hdm_episode_starts", "vm_episode_starts", "vendors_with_episode",
        "right_censored_episodes", "episode_dose_conflicts", "episode_duration_mismatches",
    ]
    for column in episode_count_columns:
        if column not in result:
            result[column] = 0
        result[column] = result[column].fillna(0)
    result["vm_episode_starts_per_1000_orders"] = safe_divide(
        result.vm_episode_starts, result.orders, 1000
    )
    return result


def pre_post_table(summary, dimensions):
    dimensions = list(dimensions)
    available_metrics = [metric for metric in SUMMARY_METRICS if metric in summary.columns]
    wide = summary.pivot(index=dimensions, columns="period", values=available_metrics)
    wide.columns = [f"{metric}_{period.lower()}" for metric, period in wide.columns]
    wide = wide.reset_index()
    for metric in available_metrics:
        pre_col, post_col = f"{metric}_pre", f"{metric}_post"
        if pre_col not in wide:
            wide[pre_col] = np.nan
        if post_col not in wide:
            wide[post_col] = np.nan
        wide[f"{metric}_delta"] = wide[post_col] - wide[pre_col]
    return wide


df_summary_change_long = build_summary(
    df_order_analysis, df_episode_analysis, ["comparison_id"]
)
df_summary_change_grade_long = build_summary(
    df_order_analysis, df_episode_analysis, ["comparison_id", "vendor_grade"]
)

df_order_analysis_flag = (
    df_order_analysis.explode("analysis_flags").rename(columns={"analysis_flags": "analysis_flag"})
)
df_episode_analysis_flag = (
    df_episode_analysis.explode("analysis_flags").rename(columns={"analysis_flags": "analysis_flag"})
)
df_summary_change_flag_long = build_summary(
    df_order_analysis_flag, df_episode_analysis_flag, ["comparison_id", "analysis_flag"]
)
df_summary_change_grade_flag_long = build_summary(
    df_order_analysis_flag,
    df_episode_analysis_flag,
    ["comparison_id", "vendor_grade", "analysis_flag"],
)

comparison_change = pre_post_table(df_summary_change_long, ["comparison_id"])
comparison_change_grade = pre_post_table(
    df_summary_change_grade_long, ["comparison_id", "vendor_grade"]
)
comparison_change_flag = pre_post_table(
    df_summary_change_flag_long, ["comparison_id", "analysis_flag"]
)
comparison_change_grade_flag = pre_post_table(
    df_summary_change_grade_flag_long,
    ["comparison_id", "vendor_grade", "analysis_flag"],
)

table_metadata = df_config[[
    "comparison_id", "comparison_label", "sheet_row", "implementation_date",
    "pre_start", "pre_end", "post_start", "post_end", "days_per_period",
    "eligible_vendors", "eligible_franchises", "trigger_ids_involved",
]]
for table_name in [
    "comparison_change", "comparison_change_grade",
    "comparison_change_flag", "comparison_change_grade_flag",
]:
    table = globals()[table_name].merge(
        table_metadata, on="comparison_id", how="left", validate="many_to_one"
    )
    leading = list(table_metadata.columns)
    remaining = [column for column in table.columns if column not in leading]
    table = table[leading + remaining].copy()
    numeric_columns = table.select_dtypes(include="number").columns
    table[numeric_columns] = table[numeric_columns].round(3)
    globals()[table_name] = table
    print(table_name)
    display(globals()[table_name])


comparison_change


,comparison_id,comparison_label,sheet_row,implementation_date,pre_start,pre_end,post_start,post_end,days_per_period,eligible_vendors,...,reason_coverage_pct_delta,stack_coverage_pct_delta,hdm_episode_starts_delta,vm_episode_starts_delta,vm_episode_starts_per_1000_orders_delta,vm_episode_duration_avg_min_delta,vm_episode_duration_p50_min_delta,vm_episode_duration_p90_min_delta,vm_episode_minutes_total_delta,vm_episode_dose_avg_min_delta
0,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.703131,3486.0,3454.0,25.194427,7.748339,-8.9,0.0,167979.816667,0.909131
1,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,0.00091,-0.66685,839.0,869.0,6.190541,61.470025,-5.0,363.708333,1017270.216667,0.675518
2,change_row_4,2026-09-04 · se agrega una capa extra · fila 4,4,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,65601,...,0.003213,-0.647804,621.0,697.0,18.160332,220.03501,826.533333,0.0,1113848.05,0.366985
3,change_row_5,2026-08-24 · NIU sushi: sus triggers bajan la ...,5,2026-08-24,2026-08-10,2026-08-23,2026-08-24,2026-09-06,14,63,...,0.0,0.72431,637.0,640.0,26.926032,-1.520814,-2.283333,0.0,8262.95,1.176216


comparison_change_grade


,comparison_id,comparison_label,sheet_row,implementation_date,pre_start,pre_end,post_start,post_end,days_per_period,eligible_vendors,...,reason_coverage_pct_delta,stack_coverage_pct_delta,hdm_episode_starts_delta,vm_episode_starts_delta,vm_episode_starts_per_1000_orders_delta,vm_episode_duration_avg_min_delta,vm_episode_duration_p50_min_delta,vm_episode_duration_p90_min_delta,vm_episode_minutes_total_delta,vm_episode_dose_avg_min_delta
0,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.636831,2093.0,2061.0,23.812215,12.935196,-5.0,0.0,132709.233333,0.904018
1,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.678399,877.0,878.0,26.516532,3.190118,-6.558333,0.0,25971.083333,0.980151
2,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-1.141296,516.0,515.0,28.724825,-2.28772,-4.733333,0.0,9299.5,0.830425
3,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,-0.00108,-0.769301,420.0,437.0,6.891195,17.892926,-5.0,0.0,169213.166667,0.70957
4,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,0.002593,-0.582073,419.0,432.0,5.570233,113.83232,0.0,776.338333,848057.05,0.63515
5,change_row_4,2026-09-04 · se agrega una capa extra · fila 4,4,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,65601,...,0.003292,-0.65006,692.0,759.0,19.002129,229.388336,833.883333,0.0,1169539.95,0.381731
6,change_row_4,2026-09-04 · se agrega una capa extra · fila 4,4,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,65601,...,0.0,-3.719203,-71.0,-62.0,-37.31441,-167.472753,-1052.266667,0.0,-55691.9,-0.505523
7,change_row_5,2026-08-24 · NIU sushi: sus triggers bajan la ...,5,2026-08-24,2026-08-10,2026-08-23,2026-08-24,2026-09-06,14,63,...,0.0,-0.429239,423.0,426.0,32.375935,-2.037826,-3.975,0.0,4709.516667,1.104709
8,change_row_5,2026-08-24 · NIU sushi: sus triggers bajan la ...,5,2026-08-24,2026-08-10,2026-08-23,2026-08-24,2026-09-06,14,63,...,0.0,0.078127,25.0,25.0,3.74446,-1.185505,-4.633333,0.0,107.683333,0.922976
9,change_row_5,2026-08-24 · NIU sushi: sus triggers bajan la ...,5,2026-08-24,2026-08-10,2026-08-23,2026-08-24,2026-09-06,14,63,...,0.0,2.326007,173.0,173.0,24.524704,-0.737395,0.0,0.0,3021.966667,1.556939


comparison_change_flag


,comparison_id,comparison_label,sheet_row,implementation_date,pre_start,pre_end,post_start,post_end,days_per_period,eligible_vendors,...,reason_coverage_pct_delta,stack_coverage_pct_delta,hdm_episode_starts_delta,vm_episode_starts_delta,vm_episode_starts_per_1000_orders_delta,vm_episode_duration_avg_min_delta,vm_episode_duration_p50_min_delta,vm_episode_duration_p90_min_delta,vm_episode_minutes_total_delta,vm_episode_dose_avg_min_delta
0,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-1.003977,-32.0,-53.0,-0.341477,57.98531,0.0,0.0,82056.166667,0.003624
1,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.47592,3518.0,3507.0,40.311634,-1.482361,-10.0,-5.0,85923.65,1.073808
2,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,0.000998,-0.225183,619.0,611.0,7.441966,92.056089,0.0,720.955,482541.3,0.002151
3,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,0.000851,-1.043511,220.0,258.0,5.434059,47.772126,-9.158333,274.708333,534728.916667,0.974416
4,change_row_4,2026-09-04 · se agrega una capa extra · fila 4,4,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,65601,...,-0.000327,-0.217108,194.0,295.0,13.500178,156.554263,680.65,0.0,339907.416667,-0.146177
5,change_row_4,2026-09-04 · se agrega una capa extra · fila 4,4,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,65601,...,0.006628,-1.071431,427.0,402.0,23.466559,244.031264,863.566667,0.0,773940.633333,0.589897
6,change_row_5,2026-08-24 · NIU sushi: sus triggers bajan la ...,5,2026-08-24,2026-08-10,2026-08-23,2026-08-24,2026-09-06,14,63,...,0.0,-0.664565,-1.0,-1.0,-1.975675,-1.047604,-2.616667,-1.663333,-352.95,0.141281
7,change_row_5,2026-08-24 · NIU sushi: sus triggers bajan la ...,5,2026-08-24,2026-08-10,2026-08-23,2026-08-24,2026-09-06,14,63,...,0.0,1.038365,638.0,641.0,33.515273,-1.619795,-2.183333,0.0,8615.9,1.311368


comparison_change_grade_flag


,comparison_id,comparison_label,sheet_row,implementation_date,pre_start,pre_end,post_start,post_end,days_per_period,eligible_vendors,...,reason_coverage_pct_delta,stack_coverage_pct_delta,hdm_episode_starts_delta,vm_episode_starts_delta,vm_episode_starts_per_1000_orders_delta,vm_episode_duration_avg_min_delta,vm_episode_duration_p50_min_delta,vm_episode_duration_p90_min_delta,vm_episode_minutes_total_delta,vm_episode_dose_avg_min_delta
0,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.770573,20.0,0.0,0.695863,76.964607,0.0,254.891667,67267.066667,-0.009153
1,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.509205,2073.0,2061.0,39.441617,0.593848,-5.0,-5.0,65442.166667,1.081036
2,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-1.303028,-10.0,-9.0,-0.38001,32.705249,0.8,0.0,10774.95,0.044744
3,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.193269,887.0,887.0,42.814248,-1.896458,-5.666667,-5.0,15196.133333,1.135398
4,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-2.433543,-42.0,-44.0,-11.690343,20.014633,4.966667,0.0,4014.15,0.015053
5,change_row_2,2026-09-04 · se agrega una capa extra · fila 2,2,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,538,...,0.0,-0.779759,558.0,559.0,39.098153,-7.234632,-6.2,-5.0,5285.35,0.968764
6,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,-0.002516,-0.535624,305.0,290.0,8.90225,39.281909,0.0,18.646667,111907.0,0.005775
7,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,0.0,-0.937033,115.0,147.0,5.091711,8.672819,-10.0,0.0,57306.166667,1.00822
8,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,0.003742,0.035517,314.0,321.0,6.476963,150.763033,0.0,863.928333,370634.3,0.00021
9,change_row_3,2026-09-04 · se agrega una capa extra · fila 3,3,2026-09-04,2026-08-28,2026-09-02,2026-09-04,2026-09-09,6,4184,...,0.001607,-1.160515,105.0,111.0,5.402651,96.547852,-5.0,733.555,477422.75,0.932168


In [ ]:
def plot_overview(comparison_id):
    meta = df_config.set_index("comparison_id").loc[comparison_id]
    data = (
        df_summary_change_long[df_summary_change_long.comparison_id.eq(comparison_id)]
        .set_index("period")
        .reindex(["PRE", "POST"])
    )
    if data.stack_known_orders.fillna(0).eq(0).any():
        print(f"{comparison_id}: no hay base común EPT/HDM/AWT para dibujar el stack.")
        return
    fig = go.Figure()
    for field, label, color in [
        ("stack_ept_base_avg_min", "EPT sin HDM", "#3B82F6"),
        ("stack_hdm_avg_min", "HDM añadido", "#F59E0B"),
        ("stack_awt_avg_min", "AWT", "#8B5CF6"),
    ]:
        fig.add_bar(
            x=["PRE", "POST"], y=data[field], name=label, marker_color=color,
            text=data[field].map(lambda value: f"{value:.2f}"), textposition="inside",
        )
    fig.update_layout(
        template="plotly_white", barmode="stack", height=480,
        title=(
            f"{meta.comparison_label}<br><sup>{meta.days_per_period} días por período · "
            f"misma base de órdenes con EPT, HDM y AWT conocidos</sup>"
        ),
        yaxis_title="minutos promedio por orden", legend_orientation="h",
    )
    fig.show()


for comparison_id in df_config.comparison_id:
    plot_overview(comparison_id)


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

: 

## 5. Distribuciones CEIL de AWT y HDM

- AWT CEIL 5 significa `4 < AWT <= 5` minutos. Se reportan órdenes, share sobre el total, share sobre AWT conocido, volumen de minutos y share del volumen.
- HDM CEIL usa los minutos añadidos. Se reporta share sobre todas las órdenes y share dentro de las órdenes afectadas.
- Las tablas conservan cada CEIL exacto. Sólo los gráficos agrupan la cola configurada como `20+`.


In [ ]:
def add_pre_post_deltas(long_table, index_columns, value_columns):
    wide = long_table.pivot(index=index_columns, columns="period", values=value_columns)
    wide.columns = [f"{metric}_{period.lower()}" for metric, period in wide.columns]
    wide = wide.reset_index()
    for metric in value_columns:
        pre_col, post_col = f"{metric}_pre", f"{metric}_post"
        if pre_col not in wide:
            wide[pre_col] = np.nan
        if post_col not in wide:
            wide[post_col] = np.nan
        wide[f"{metric}_delta"] = wide[post_col] - wide[pre_col]
    return wide


distribution_totals = (
    df_order_analysis.groupby(["comparison_id", "period"], observed=True)
    .agg(
        total_orders=("orders", "sum"),
        total_awt_known_orders=("awt_known_orders", "sum"),
        total_awt_minutes=("awt_minutes", "sum"),
        total_hdm_orders=("hdm_orders", "sum"),
        total_hdm_minutes=("hdm_minutes", "sum"),
    )
    .reset_index()
)

df_awt_distribution_long = (
    df_order_analysis[df_order_analysis.awt_ceil_min.notna()]
    .groupby(["comparison_id", "period", "awt_ceil_min"], observed=True, dropna=False)
    .agg(bucket_orders=("awt_known_orders", "sum"), bucket_minutes=("awt_minutes", "sum"))
    .reset_index()
    .merge(distribution_totals, on=["comparison_id", "period"], validate="many_to_one")
)
df_awt_distribution_long["share_all_orders_pct"] = safe_divide(
    df_awt_distribution_long.bucket_orders, df_awt_distribution_long.total_orders, 100
)
df_awt_distribution_long["share_awt_known_pct"] = safe_divide(
    df_awt_distribution_long.bucket_orders,
    df_awt_distribution_long.total_awt_known_orders,
    100,
)
df_awt_distribution_long["share_awt_minutes_pct"] = safe_divide(
    df_awt_distribution_long.bucket_minutes,
    df_awt_distribution_long.total_awt_minutes,
    100,
)

df_hdm_distribution_long = (
    df_order_analysis[df_order_analysis.hdm_ceil_min.gt(0)]
    .groupby(["comparison_id", "period", "hdm_ceil_min"], observed=True, dropna=False)
    .agg(bucket_orders=("hdm_orders", "sum"), bucket_minutes=("hdm_minutes", "sum"))
    .reset_index()
    .merge(distribution_totals, on=["comparison_id", "period"], validate="many_to_one")
)
df_hdm_distribution_long["share_all_orders_pct"] = safe_divide(
    df_hdm_distribution_long.bucket_orders, df_hdm_distribution_long.total_orders, 100
)
df_hdm_distribution_long["share_hdm_orders_pct"] = safe_divide(
    df_hdm_distribution_long.bucket_orders, df_hdm_distribution_long.total_hdm_orders, 100
)
df_hdm_distribution_long["share_hdm_minutes_pct"] = safe_divide(
    df_hdm_distribution_long.bucket_minutes, df_hdm_distribution_long.total_hdm_minutes, 100
)

awt_distribution = add_pre_post_deltas(
    df_awt_distribution_long,
    ["comparison_id", "awt_ceil_min"],
    ["bucket_orders", "bucket_minutes", "share_all_orders_pct", "share_awt_known_pct", "share_awt_minutes_pct"],
).round(3)
hdm_distribution = add_pre_post_deltas(
    df_hdm_distribution_long,
    ["comparison_id", "hdm_ceil_min"],
    ["bucket_orders", "bucket_minutes", "share_all_orders_pct", "share_hdm_orders_pct", "share_hdm_minutes_pct"],
).round(3)

print("AWT CEIL — ejemplo: filtrar awt_ceil_min == 5")
display(awt_distribution)
print("HDM minutos añadidos")
display(hdm_distribution)


In [ ]:
def collapse_plot_tail(data, bucket_column, cap=PLOT_BUCKET_CAP):
    plot_data = data.copy()
    plot_data["bucket_label"] = plot_data[bucket_column].map(
        lambda value: f"{cap}+" if value >= cap else str(int(value))
    )
    plot_data["bucket_sort"] = plot_data[bucket_column].clip(upper=cap)
    grouped = (
        plot_data.groupby(
            ["comparison_id", "period", "bucket_label", "bucket_sort"], observed=True
        )
        .agg(
            bucket_orders=("bucket_orders", "sum"),
            bucket_minutes=("bucket_minutes", "sum"),
            total_orders=("total_orders", "first"),
        )
        .reset_index()
    )
    grouped["share_all_orders_pct"] = safe_divide(
        grouped.bucket_orders, grouped.total_orders, 100
    )
    grouped["minute_share_pct"] = grouped.groupby(
        ["comparison_id", "period"], observed=True
    ).bucket_minutes.transform(lambda values: 100 * values / values.sum() if values.sum() else np.nan)
    return grouped.sort_values("bucket_sort")


def plot_distributions(comparison_id):
    meta = df_config.set_index("comparison_id").loc[comparison_id]
    awt = collapse_plot_tail(df_awt_distribution_long, "awt_ceil_min")
    hdm = collapse_plot_tail(df_hdm_distribution_long, "hdm_ceil_min")
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            "AWT · share de órdenes totales", "AWT · share del volumen de minutos",
            "HDM · share de órdenes totales", "HDM · share de minutos añadidos",
        ],
    )
    colors = {"PRE": "#64748B", "POST": "#0F766E"}
    for row, dataset in [(1, awt), (2, hdm)]:
        part = dataset[dataset.comparison_id.eq(comparison_id)]
        for period in ["PRE", "POST"]:
            p = part[part.period.eq(period)]
            custom = np.column_stack([p.bucket_orders, p.bucket_minutes]) if len(p) else None
            fig.add_bar(
                x=p.bucket_label, y=p.share_all_orders_pct,
                name=period, legendgroup=period, showlegend=(row == 1),
                marker_color=colors[period], customdata=custom,
                hovertemplate=(
                    "CEIL %{x}<br>% órdenes totales: %{y:.2f}%"
                    "<br>órdenes: %{customdata[0]:,.0f}<br>minutos: %{customdata[1]:,.1f}"
                    f"<extra>{period}</extra>"
                ), row=row, col=1,
            )
            fig.add_bar(
                x=p.bucket_label, y=p.minute_share_pct,
                name=period, legendgroup=period, showlegend=False,
                marker_color=colors[period], customdata=custom,
                hovertemplate=(
                    "CEIL %{x}<br>% volumen minutos: %{y:.2f}%"
                    "<br>órdenes: %{customdata[0]:,.0f}<br>minutos: %{customdata[1]:,.1f}"
                    f"<extra>{period}</extra>"
                ), row=row, col=2,
            )
    fig.update_layout(
        template="plotly_white", barmode="group", height=760,
        title=f"Distribuciones PRE/POST · {meta.comparison_label}",
        legend_orientation="h",
    )
    fig.update_xaxes(title_text="bucket CEIL (min)")
    fig.update_yaxes(title_text="%")
    fig.show()


for comparison_id in df_config.comparison_id:
    plot_distributions(comparison_id)


## 6. Heatmaps por weekday y bloque

Los tres paneles de cada métrica son PRE, POST y delta POST−PRE. Las celdas con menos de `MIN_ORDERS_HEATMAP` órdenes se enmascaran sólo en el gráfico; la tabla exportada conserva los valores y el volumen.


In [ ]:
df_daypart_long = aggregate_orders(
    df_order_analysis[df_order_analysis.mealpart.isin(MEALPART_ORDER)],
    ["comparison_id", "weekday", "mealpart"],
)
DAYPART_METRICS = [
    "awt_avg_min",
    "awt_gt10_share_pct",
    "hdm_frequency_pct",
    "hdm_intensity_min",
    "hdm_minutes_per_100_orders",
]
daypart_heatmap = add_pre_post_deltas(
    df_daypart_long,
    ["comparison_id", "weekday", "mealpart"],
    ["orders"] + DAYPART_METRICS,
).round(3)
display(daypart_heatmap)


def heatmap_matrix(data, value_column):
    return (
        data.pivot(index="mealpart", columns="weekday", values=value_column)
        .reindex(index=MEALPART_ORDER, columns=WEEKDAY_ORDER)
    )


def plot_daypart_heatmaps(comparison_id):
    meta = df_config.set_index("comparison_id").loc[comparison_id]
    labels = {
        "awt_avg_min": "AWT promedio (min)",
        "awt_gt10_share_pct": "Órdenes AWT >10 (%)",
        "hdm_frequency_pct": "Órdenes con HDM (%)",
        "hdm_intensity_min": "HDM promedio entre afectadas (min)",
        "hdm_minutes_per_100_orders": "Minutos HDM / 100 órdenes",
    }
    fig = make_subplots(
        rows=len(DAYPART_METRICS), cols=3,
        subplot_titles=[
            f"{labels[metric]} · {view}"
            for metric in DAYPART_METRICS for view in ("PRE", "POST", "Δ")
        ],
        vertical_spacing=0.06,
    )
    base = df_daypart_long[df_daypart_long.comparison_id.eq(comparison_id)]
    for row, metric in enumerate(DAYPART_METRICS, start=1):
        matrices = {}
        order_matrices = {}
        for period in ["PRE", "POST"]:
            part = base[base.period.eq(period)].copy()
            values = heatmap_matrix(part, metric)
            orders = heatmap_matrix(part, "orders")
            matrices[period] = values.where(orders >= MIN_ORDERS_HEATMAP)
            order_matrices[period] = orders
        delta = matrices["POST"] - matrices["PRE"]
        pre_values = matrices["PRE"].to_numpy(dtype=float, na_value=np.nan)
        post_values = matrices["POST"].to_numpy(dtype=float, na_value=np.nan)
        delta_values = delta.to_numpy(dtype=float, na_value=np.nan)
        common_values = np.concatenate([
            pre_values.ravel(), post_values.ravel()
        ])
        finite_common = common_values[np.isfinite(common_values)]
        zmin = float(finite_common.min()) if finite_common.size else None
        zmax = float(finite_common.max()) if finite_common.size else None
        finite_delta = np.abs(delta_values.ravel()[np.isfinite(delta_values.ravel())])
        delta_limit = float(finite_delta.max()) if finite_delta.size else 1.0

        for col, (view, matrix) in enumerate(
            [("PRE", matrices["PRE"]), ("POST", matrices["POST"]), ("DELTA", delta)],
            start=1,
        ):
            custom_orders = (
                order_matrices[view] if view in order_matrices
                else np.minimum(
                    order_matrices["PRE"].to_numpy(dtype=float, na_value=np.nan),
                    order_matrices["POST"].to_numpy(dtype=float, na_value=np.nan),
                )
            )
            z_values = matrix.to_numpy(dtype=float, na_value=np.nan)
            custom_values = (
                custom_orders.to_numpy(dtype=float, na_value=np.nan)
                if isinstance(custom_orders, pd.DataFrame)
                else np.asarray(custom_orders, dtype=float)
            )
            fig.add_trace(
                go.Heatmap(
                    z=z_values, x=matrix.columns, y=matrix.index,
                    colorscale="RdBu" if view == "DELTA" else "Blues",
                    zmid=0 if view == "DELTA" else None,
                    zmin=-delta_limit if view == "DELTA" else zmin,
                    zmax=delta_limit if view == "DELTA" else zmax,
                    text=np.round(z_values, 2), texttemplate="%{text}",
                    customdata=custom_values,
                    hovertemplate=(
                        "weekday: %{x}<br>bloque: %{y}<br>valor: %{z:.2f}"
                        "<br>órdenes/base mínima: %{customdata:,.0f}<extra></extra>"
                    ),
                    showscale=False,
                ), row=row, col=col,
            )
    fig.update_layout(
        template="plotly_white", height=260 * len(DAYPART_METRICS),
        title=(
            f"Weekday × bloque · {meta.comparison_label}<br>"
            f"<sup>breakfast 08–10 · lunch 12–14 · dinner 19–21 · mínimo {MIN_ORDERS_HEATMAP} órdenes</sup>"
        ),
    )
    fig.show()


for comparison_id in df_config.comparison_id:
    plot_daypart_heatmaps(comparison_id)


## 7. Mix horario de dosis HDM

`share_among_hdm_orders_pct` responde directamente preguntas como: “martes 18:00, ¿qué porcentaje de las órdenes con HDM tuvo +5, +10, etc.?”. El denominador son sólo las órdenes con HDM positivo; `share_all_orders_pct` muestra aparte el alcance sobre el total.


In [ ]:
hourly_totals = (
    df_order_analysis.groupby(
        ["comparison_id", "period", "weekday", "order_hour"], observed=True
    )
    .agg(
        all_orders=("orders", "sum"),
        all_hdm_orders=("hdm_orders", "sum"),
        all_hdm_minutes=("hdm_minutes", "sum"),
    )
    .reset_index()
)
hourly_hdm_mix = (
    df_order_analysis[df_order_analysis.hdm_ceil_min.gt(0)]
    .groupby(
        ["comparison_id", "period", "weekday", "order_hour", "hdm_ceil_min"],
        observed=True,
    )
    .agg(bucket_orders=("hdm_orders", "sum"), bucket_minutes=("hdm_minutes", "sum"))
    .reset_index()
    .merge(
        hourly_totals,
        on=["comparison_id", "period", "weekday", "order_hour"],
        validate="many_to_one",
    )
)
hourly_hdm_mix["share_among_hdm_orders_pct"] = safe_divide(
    hourly_hdm_mix.bucket_orders, hourly_hdm_mix.all_hdm_orders, 100
)
hourly_hdm_mix["share_all_orders_pct"] = safe_divide(
    hourly_hdm_mix.bucket_orders, hourly_hdm_mix.all_orders, 100
)
hourly_hdm_mix["share_hdm_minutes_pct"] = safe_divide(
    hourly_hdm_mix.bucket_minutes, hourly_hdm_mix.all_hdm_minutes, 100
)
period_minutes = hourly_hdm_mix.groupby(
    ["comparison_id", "period"], observed=True
).bucket_minutes.transform("sum")
hourly_hdm_mix["temporal_minute_share_pct"] = safe_divide(
    hourly_hdm_mix.bucket_minutes, period_minutes, 100
)
display(hourly_hdm_mix)


def plot_hourly_hdm_mix(comparison_id, weekday):
    data = hourly_hdm_mix[
        hourly_hdm_mix.comparison_id.eq(comparison_id)
        & hourly_hdm_mix.weekday.astype(str).eq(weekday)
    ].copy()
    if data.empty:
        print(f"{comparison_id} · {weekday}: sin órdenes HDM.")
        return
    meta = df_config.set_index("comparison_id").loc[comparison_id]
    data["dose_label"] = data.hdm_ceil_min.map(
        lambda value: f"+{PLOT_BUCKET_CAP}+" if value >= PLOT_BUCKET_CAP else f"+{int(value)}"
    )
    data["dose_sort"] = data.hdm_ceil_min.clip(upper=PLOT_BUCKET_CAP)
    data = (
        data.groupby(
            ["period", "order_hour", "dose_label", "dose_sort"], observed=True
        )
        .agg(
            bucket_orders=("bucket_orders", "sum"),
            all_hdm_orders=("all_hdm_orders", "first"),
            all_orders=("all_orders", "first"),
        )
        .reset_index()
        .sort_values("dose_sort")
    )
    data["mix_pct"] = safe_divide(data.bucket_orders, data.all_hdm_orders, 100)
    data["hdm_prevalence_pct"] = safe_divide(data.all_hdm_orders, data.all_orders, 100)
    fig = make_subplots(rows=1, cols=2, subplot_titles=["PRE", "POST"], shared_yaxes=True)
    dose_labels = data[["dose_label", "dose_sort"]].drop_duplicates().sort_values("dose_sort")
    palette = px.colors.qualitative.Safe + px.colors.qualitative.Set3
    for index, dose in enumerate(dose_labels.dose_label):
        for col, period in enumerate(["PRE", "POST"], start=1):
            part = data[(data.period.eq(period)) & (data.dose_label.eq(dose))]
            custom = np.column_stack([
                part.bucket_orders, part.all_hdm_orders, part.all_orders, part.hdm_prevalence_pct
            ]) if len(part) else None
            fig.add_bar(
                x=part.order_hour, y=part.mix_pct, name=dose,
                legendgroup=dose, showlegend=(col == 1), marker_color=palette[index % len(palette)],
                customdata=custom,
                hovertemplate=(
                    "hora %{x}:00<br>dosis " + dose + ": %{y:.1f}% de órdenes HDM"
                    "<br>órdenes bucket: %{customdata[0]:,.0f}"
                    "<br>órdenes HDM: %{customdata[1]:,.0f}"
                    "<br>órdenes totales: %{customdata[2]:,.0f}"
                    "<br>prevalencia HDM: %{customdata[3]:.1f}%<extra></extra>"
                ), row=1, col=col,
            )
    fig.update_layout(
        template="plotly_white", barmode="stack", height=520,
        title=f"Mix de minutos HDM · {weekday} · {meta.comparison_label}",
        yaxis_title="% dentro de órdenes con HDM", legend_title="dosis",
    )
    fig.update_xaxes(title_text="hora local", dtick=1)
    fig.update_yaxes(range=[0, 100])
    fig.show()


for comparison_id in df_config.comparison_id:
    for weekday in HOURLY_MIX_WEEKDAYS:
        plot_hourly_hdm_mix(comparison_id, weekday)


## 8. Frecuencia, duración y horario de episodios

El autor `log_vendor_monitor` es el proxy más cercano a activaciones automáticas. Se muestran todos los autores en las tablas, pero los KPIs principales de duración usan Vendor Monitor. `duration_difference_min` audita la equivalencia entre el campo crudo (segundos / 60) y la diferencia de timestamps.


In [ ]:
episode_dimensions = [
    "comparison_id", "period", "weekday", "enabled_hour_local", "mealpart",
    "author", "minutes_added",
]
trigger_episode_detail = df_episode_analysis[[
    "comparison_id", "period", "vendor_code", "vendor_name", "franchise_id",
    "franchise_name", "vendor_grade", "enabled_at_local", "disabled_at",
    "weekday", "enabled_hour_local", "mealpart", "author", "minutes_added",
    "duration_min", "reported_duration_min", "timestamp_duration_min",
    "duration_difference_min", "observed_orders", "right_censored",
    "dose_variants", "author_variants",
]].copy()

trigger_episode_summary = (
    df_episode_analysis.groupby(episode_dimensions, observed=True, dropna=False)
    .agg(
        episode_starts=("enabled_at", "count"),
        vendors=("vendor_code", "nunique"),
        observed_orders=("observed_orders", "sum"),
        duration_avg_min=("duration_min", "mean"),
        duration_p50_min=("duration_min", "median"),
        duration_p90_min=("duration_min", lambda values: values.quantile(0.90)),
        right_censored=("right_censored", "sum"),
    )
    .reset_index()
    .round(3)
)
display(trigger_episode_summary)


def plot_activation_heatmap(comparison_id, vendor_monitor_only=True):
    data = df_episode_analysis[df_episode_analysis.comparison_id.eq(comparison_id)].copy()
    if vendor_monitor_only:
        data = data[data.is_vendor_monitor]
    if data.empty:
        print(f"{comparison_id}: sin episodios para el filtro elegido.")
        return
    meta = df_config.set_index("comparison_id").loc[comparison_id]
    counts = (
        data.groupby(["period", "weekday", "enabled_hour_local"], observed=True)
        .size().rename("starts").reset_index()
    )
    matrices = {}
    for period in ["PRE", "POST"]:
        matrices[period] = (
            counts[counts.period.eq(period)]
            .pivot(index="weekday", columns="enabled_hour_local", values="starts")
            .reindex(index=WEEKDAY_ORDER, columns=range(24))
            .fillna(0)
        )
    matrices["DELTA"] = matrices["POST"] - matrices["PRE"]
    fig = make_subplots(rows=1, cols=3, subplot_titles=["PRE", "POST", "Δ POST−PRE"])
    max_count = max(matrices["PRE"].to_numpy().max(), matrices["POST"].to_numpy().max(), 1)
    delta_limit = max(np.abs(matrices["DELTA"].to_numpy()).max(), 1)
    for col, view in enumerate(["PRE", "POST", "DELTA"], start=1):
        matrix = matrices[view]
        fig.add_trace(go.Heatmap(
            z=matrix.values, x=matrix.columns, y=matrix.index,
            colorscale="RdBu" if view == "DELTA" else "Blues",
            zmid=0 if view == "DELTA" else None,
            zmin=-delta_limit if view == "DELTA" else 0,
            zmax=delta_limit if view == "DELTA" else max_count,
            text=matrix.values.astype(int), texttemplate="%{text}",
            hovertemplate="%{y} %{x}:00<br>inicios: %{z}<extra></extra>",
            showscale=False,
        ), row=1, col=col)
    source = "Vendor Monitor" if vendor_monitor_only else "todos los autores"
    fig.update_layout(
        template="plotly_white", height=520,
        title=f"Inicios de episodios por hora · {source} · {meta.comparison_label}",
    )
    fig.update_xaxes(title_text="hora local", dtick=2)
    fig.show()


def plot_episode_duration(comparison_id):
    data = df_episode_analysis[
        df_episode_analysis.comparison_id.eq(comparison_id)
        & df_episode_analysis.is_vendor_monitor
        & df_episode_analysis.duration_min.notna()
    ].copy()
    if data.empty:
        return
    meta = df_config.set_index("comparison_id").loc[comparison_id]
    fig = px.box(
        data, x="period", y="duration_min", color="period", points=False,
        category_orders={"period": ["PRE", "POST"]},
        color_discrete_map={"PRE": "#64748B", "POST": "#0F766E"},
        title=f"Duración de episodios Vendor Monitor · {meta.comparison_label}",
    )
    fig.update_layout(template="plotly_white", showlegend=False, height=450)
    fig.update_yaxes(title="duración (min)")
    fig.show()


def plot_episode_timeline(comparison_id, period="POST", max_vendors=TIMELINE_MAX_VENDORS):
    data = df_episode_analysis[
        df_episode_analysis.comparison_id.eq(comparison_id)
        & df_episode_analysis.period.eq(period)
        & df_episode_analysis.is_vendor_monitor
        & df_episode_analysis.disabled_at.notna()
    ].copy()
    if data.empty:
        print(f"{comparison_id} · {period}: sin episodios cerrados para timeline.")
        return
    top_vendors = data.groupby("vendor_code").size().nlargest(max_vendors).index
    data = data[data.vendor_code.isin(top_vendors)].copy()
    data["dose"] = data.minutes_added.map(lambda value: f"+{value:g}" if pd.notna(value) else "conflicto")
    fig = px.timeline(
        data, x_start="enabled_at", x_end="disabled_at", y="vendor_code", color="dose",
        hover_data=["vendor_name", "franchise_name", "duration_min", "observed_orders"],
        title=f"Timeline {period} · top {len(top_vendors)} vendors por episodios",
    )
    fig.update_layout(template="plotly_white", height=max(500, 22 * len(top_vendors)))
    fig.show()


for comparison_id in df_config.comparison_id:
    plot_activation_heatmap(comparison_id)
    plot_episode_duration(comparison_id)

# Timeline bajo demanda; descomentar una comparación si se necesita detalle operacional.
# plot_episode_timeline(df_config.comparison_id.iloc[0], period="POST")


## 9. Relación dosis HDM × AWT

Esta matriz adicional muestra, dentro de cada dosis HDM, cómo se distribuyen los buckets AWT. Es descriptiva: HDM suele activarse precisamente cuando el vendor ya está deteriorado, por lo que no debe leerse como efecto causal aislado.


In [ ]:
hdm_awt_matrix = (
    df_order_analysis[
        df_order_analysis.hdm_ceil_min.gt(0)
        & df_order_analysis.awt_ceil_min.notna()
    ]
    .groupby(
        ["comparison_id", "period", "hdm_ceil_min", "awt_ceil_min"], observed=True
    )
    .agg(orders=("orders", "sum"))
    .reset_index()
)
hdm_awt_matrix["share_within_hdm_dose_pct"] = (
    hdm_awt_matrix.groupby(
        ["comparison_id", "period", "hdm_ceil_min"], observed=True
    ).orders.transform(lambda values: 100 * values / values.sum() if values.sum() else np.nan)
)
hdm_awt_matrix["share_within_hdm_dose_pct"] = pd.to_numeric(
    hdm_awt_matrix.share_within_hdm_dose_pct, errors="coerce"
).astype(float)
display(hdm_awt_matrix)


def plot_hdm_awt_matrix(comparison_id):
    data = hdm_awt_matrix[hdm_awt_matrix.comparison_id.eq(comparison_id)].copy()
    if data.empty:
        return
    data["awt_plot_bucket"] = data.awt_ceil_min.clip(upper=PLOT_BUCKET_CAP)
    collapsed = (
        data.groupby(
            ["period", "hdm_ceil_min", "awt_plot_bucket"], observed=True
        ).orders.sum().reset_index()
    )
    collapsed["share_pct"] = collapsed.groupby(
        ["period", "hdm_ceil_min"], observed=True
    ).orders.transform(lambda values: 100 * values / values.sum())
    collapsed["share_pct"] = pd.to_numeric(
        collapsed.share_pct, errors="coerce"
    ).astype(float)
    fig = make_subplots(rows=1, cols=2, subplot_titles=["PRE", "POST"])
    for col, period in enumerate(["PRE", "POST"], start=1):
        part = collapsed[collapsed.period.eq(period)]
        matrix = part.pivot(
            index="hdm_ceil_min", columns="awt_plot_bucket", values="share_pct"
        ).fillna(0).sort_index().astype(float)
        fig.add_trace(go.Heatmap(
            z=matrix.values, x=matrix.columns, y=matrix.index,
            colorscale="Viridis", zmin=0, zmax=100,
            text=np.round(matrix.values, 1), texttemplate="%{text}", showscale=(col == 2),
            hovertemplate="HDM +%{y}<br>AWT CEIL %{x}<br>share: %{z:.1f}%<extra></extra>",
        ), row=1, col=col)
    fig.update_layout(
        template="plotly_white", height=520,
        title="Distribución AWT dentro de cada dosis HDM",
    )
    fig.update_xaxes(title="AWT CEIL (20 = 20+ en el gráfico)")
    fig.update_yaxes(title="minutos HDM")
    fig.show()


for comparison_id in df_config.comparison_id:
    plot_hdm_awt_matrix(comparison_id)


## 10. Controles de calidad y límites de lectura

Revisar esta tabla antes de interpretar resultados. El PRE/POST controla largo y composición de weekdays, pero no clima, demanda, otros cambios operacionales ni selección endógena del trigger. Un faltante HDM no se trata como cero en la cobertura. Las vistas por flag pueden duplicar un vendor si coincide con más de un tag y no deben sumarse.


In [ ]:
QUALITY_COLUMNS = [
    "comparison_id", "period", "orders", "active_vendors", "hdm_source_coverage_pct",
    "reason_coverage_pct", "stack_coverage_pct", "missing_hdm_source_orders",
    "conflicting_hdm_orders", "negative_awt_orders", "negative_ept_base_orders",
    "multiple_primary_orders", "right_censored_episodes", "episode_dose_conflicts",
    "episode_duration_mismatches",
]
quality_checks = df_summary_change_long[
    [column for column in QUALITY_COLUMNS if column in df_summary_change_long.columns]
].copy().round(3)

period_presence = (
    df_summary_change_long.groupby("comparison_id", observed=True).period.nunique()
    .rename("periods_with_orders").reset_index()
)
quality_checks = quality_checks.merge(period_presence, on="comparison_id", how="left")
quality_checks["warning"] = np.select(
    [
        quality_checks.periods_with_orders.lt(2),
        quality_checks.conflicting_hdm_orders.gt(0),
        quality_checks.hdm_source_coverage_pct.lt(95),
        quality_checks.negative_awt_orders.gt(0),
        quality_checks.episode_duration_mismatches.gt(0),
    ],
    [
        "falta PRE o POST con órdenes",
        "hay estados HDM contradictorios",
        "cobertura HDM menor a 95%",
        "hay AWT negativo",
        "duration no coincide con enabled/disabled",
    ],
    default="OK",
)
display(quality_checks)


## 11. Publicar resultados en Google Sheets

La celda administra únicamente las pestañas listadas. Si existen las cuatro pestañas legacy con prefijo `holy_` y aún no existe su reemplazo, primero las renombra; luego reemplaza el contenido con las tablas nuevas. No borra otras pestañas.


In [ ]:
from gspread_dataframe import set_with_dataframe
from gspread.exceptions import WorksheetNotFound

LEGACY_TAB_RENAMES = {
    "holy_modification": "comparison_change",
    "holy_modification_grade": "comparison_change_grade",
    "holy_modification_flag": "comparison_change_flag",
    "holy_modification_grade_flag": "comparison_change_grade_flag",
}

TABLES_TO_EXPORT = {
    "comparison_change": comparison_change,
    "comparison_change_grade": comparison_change_grade,
    "comparison_change_flag": comparison_change_flag,
    "comparison_change_grade_flag": comparison_change_grade_flag,
    "calendar_pre_post": df_date_pairs,
    "universe_audit": df_universe_audit,
    "franchise_rules": df_franchise_rules,
    "overlap_audit": df_overlap_audit,
    "awt_distribution": awt_distribution,
    "hdm_distribution": hdm_distribution,
    "daypart_heatmap": daypart_heatmap,
    "hourly_hdm_mix": hourly_hdm_mix,
    "trigger_episode_summary": trigger_episode_summary,
    "quality_checks": quality_checks,
}
if EXPORT_EPISODE_DETAIL:
    TABLES_TO_EXPORT["trigger_episode_detail"] = trigger_episode_detail


def sheets_ready(table):
    result = table.copy()
    for column in result.columns:
        if pd.api.types.is_datetime64_any_dtype(result[column]):
            result[column] = result[column].dt.strftime("%Y-%m-%d %H:%M:%S")
        elif isinstance(result[column].dtype, pd.CategoricalDtype):
            result[column] = result[column].astype(str)
    return result.replace([np.inf, -np.inf], np.nan).fillna("")


output_spreadsheet = gc.open_by_key(OUTPUT_SHEET_ID)
existing = {worksheet.title: worksheet for worksheet in output_spreadsheet.worksheets()}
for old_name, new_name in LEGACY_TAB_RENAMES.items():
    if old_name in existing and new_name not in existing:
        existing[old_name].update_title(new_name)
        existing[new_name] = existing.pop(old_name)
        print(f"Renombrada: {old_name} → {new_name}")

for worksheet_name, table in TABLES_TO_EXPORT.items():
    export_table = sheets_ready(table)
    try:
        output_worksheet = output_spreadsheet.worksheet(worksheet_name)
        output_worksheet.clear()
    except WorksheetNotFound:
        output_worksheet = output_spreadsheet.add_worksheet(
            title=worksheet_name,
            rows=max(len(export_table) + 10, 100),
            cols=max(len(export_table.columns) + 5, 20),
        )
    set_with_dataframe(
        output_worksheet,
        export_table,
        include_index=False,
        include_column_header=True,
        resize=True,
    )
    output_worksheet.freeze(rows=1)
    print(f"OK {worksheet_name}: {len(export_table):,} × {len(export_table.columns):,}")

print(f"Exportación terminada: https://docs.google.com/spreadsheets/d/{OUTPUT_SHEET_ID}")
